In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from pathlib import Path
import glob
import pickle
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def crpic_1H(points, positions, hauteur, largeur=0.002):
    return hauteur * (largeur**2) / ((points - positions)**2 + largeur**2)

def crpic_13C(points, positions, hauteur, largeur=0.02):
    return hauteur * (largeur**2) / ((points - positions)**2 + largeur**2)

def normaliser_intensites(df):
    max_int = df['Int.'].max()
    df['Int.'] = df['Int.'] / max_int * 1000
    return df

points_1H  = np.linspace(0, 12,  30000)
points_13C = np.linspace(0, 200, 30000)

In [3]:
SOLVANTS_1H = {
    'cdcl3':  {'ppm': 7.26,  'Int.': 500},
    'ccl4':   None,
    'dmso-d6': {'ppm': 2.50, 'Int.': 700},
    'd2o': {'ppm': 4.75, 'Int.': 600}
}

SOLVANTS_13C = {
    'cdcl3':  {'ppm': 77.16, 'Int.': 500},
    'ccl4':   None,
    'dmso-d6': {'ppm': 39.52, 'Int.': 700},
    'd2o':    None
}

def charger_molecules(base_path, solvants_dict):
    molecules_dict = {}
    for dossier_solvant in glob.glob(f'{base_path}/*/'):
        nom_solvant = Path(dossier_solvant).name.lower()
        pic_solvant = solvants_dict.get(nom_solvant, None)
        for fichier in glob.glob(f'{dossier_solvant}*.csv'):
            nom = Path(fichier).stem.split('(')[0]
            try:
                df = pd.read_csv(fichier)[['ppm', 'Int.']]
                df['ppm'] = pd.to_numeric(df['ppm'], errors='coerce')
                df['Int.'] = pd.to_numeric(df['Int.'], errors='coerce')
                df.dropna(subset=['ppm', 'Int.'], inplace=True)
                df = normaliser_intensites(df)
                molecules_dict[nom] = {'df': df, 'solvant': pic_solvant}
            except Exception as e:
                print(f"Erreur {fichier}: {e}")
    return molecules_dict

molecules_1H  = charger_molecules('/content/drive/MyDrive/IA RMN/Molecules CSV 1H', SOLVANTS_1H)
molecules_13C = charger_molecules('/content/drive/MyDrive/IA RMN/Molecules CSV 13C', SOLVANTS_13C)

molecules_communes = sorted(set(molecules_1H.keys()) & set(molecules_13C.keys()))
print(f"Molécules communes : {len(molecules_communes)}")

Molécules communes : 1244


In [4]:
import pickle

model_1H  = tf.keras.models.load_model('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/5_modele_RMN1H.h5')
model_13C = tf.keras.models.load_model('/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/2_modele_RMN13C.h5')

with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/label_encoder_1H.pkl', 'rb') as f:
    le_1H = pickle.load(f)
with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 13C/label_encoder_13C.pkl', 'rb') as f:
    le_13C = pickle.load(f)

assert len(le_1H.classes_)  == model_1H.output_shape[-1],  \
    f"Mismatch 1H : {len(le_1H.classes_)} classes vs {model_1H.output_shape[-1]} sorties"
assert len(le_13C.classes_) == model_13C.output_shape[-1], \
    f"Mismatch 13C : {len(le_13C.classes_)} classes vs {model_13C.output_shape[-1]} sorties"

molecules_communes = sorted(set(le_1H.classes_) & set(le_13C.classes_))

print(f"Classes 1H         : {len(le_1H.classes_)}")
print(f"Classes 13C        : {len(le_13C.classes_)}")
print(f"Molécules communes : {len(molecules_communes)}")

idx_1H_communes  = np.array([le_1H.transform([m])[0]  for m in molecules_communes])
idx_13C_communes = np.array([le_13C.transform([m])[0] for m in molecules_communes])

Classes 1H         : 1339
Classes 13C        : 1238
Molécules communes : 1235


In [6]:
def generer_spectre_1H(df, pic_solvant, augmentation=True):
    if augmentation:
        X_rand = np.random.uniform(-0.1, 0.1)
        Y_rand = np.random.uniform(0.8, 1.2)
        largeur = np.random.uniform(0.0015, 0.004)
    else:
        X_rand, Y_rand, largeur = 0.0, 1.0, 0.002

    spectre = np.zeros_like(points_1H)
    for _, ligne in df.iterrows():
        spectre += crpic_1H(points_1H, ligne['ppm'] + X_rand,
                             ligne['Int.'] * Y_rand, largeur)

    if pic_solvant is not None and np.random.random() < 0.7:
        spectre += crpic_1H(points_1H,
                             pic_solvant['ppm'] + np.random.uniform(-0.02, 0.02),
                             pic_solvant['Int.'] * np.random.uniform(0.8, 1.2),
                             np.random.uniform(0.0015, 0.004))

    if augmentation:
        amplitude_max = np.max(spectre)
        snr = 10 ** np.random.uniform(np.log10(150), np.log10(500))
        spectre += np.random.normal(0, amplitude_max / snr, len(points_1H))

    return spectre


def generer_spectre_13C(df, pic_solvant, augmentation=True):
    if augmentation:
        X_rand = np.random.uniform(-1.0, 1.0)
        Y_rand = np.random.uniform(0.8, 1.2)
        largeur = np.random.uniform(0.015, 0.025)
    else:
        X_rand, Y_rand, largeur = 0.0, 1.0, 0.02

    spectre = np.zeros_like(points_13C)
    for _, ligne in df.iterrows():
        spectre += crpic_13C(points_13C, ligne['ppm'] + X_rand,
                              ligne['Int.'] * Y_rand, largeur)

    if pic_solvant is not None and np.random.random() < 0.7:
        spectre += crpic_13C(points_13C,
                              pic_solvant['ppm'] + np.random.uniform(-0.2, 0.2),
                              pic_solvant['Int.'] * np.random.uniform(0.8, 1.2),
                              np.random.uniform(0.015, 0.025))

    if augmentation:
        amplitude_max = np.max(spectre)
        snr = 10 ** np.random.uniform(np.log10(150), np.log10(500))
        spectre += np.random.normal(0, amplitude_max / snr, len(points_13C))

    return spectre

In [15]:
def predire_fusion(csv_1H, csv_13C, solvant='cdcl3',
                   poids_1H=0.5, poids_13C=0.5, n_essais=10):

    solvant = solvant.lower()
    pic_solvant_1H  = SOLVANTS_1H.get(solvant, None)
    pic_solvant_13C = SOLVANTS_13C.get(solvant, None)

    df_1H = pd.read_csv(csv_1H)[['ppm', 'Int.']]
    df_1H['ppm'] = pd.to_numeric(df_1H['ppm'], errors='coerce')
    df_1H['Int.'] = pd.to_numeric(df_1H['Int.'], errors='coerce')
    df_1H.dropna(inplace=True)
    df_1H = normaliser_intensites(df_1H)

    df_13C = pd.read_csv(csv_13C)[['ppm', 'Int.']]
    df_13C['ppm'] = pd.to_numeric(df_13C['ppm'], errors='coerce')
    df_13C['Int.'] = pd.to_numeric(df_13C['Int.'], errors='coerce')
    df_13C.dropna(inplace=True)
    df_13C = normaliser_intensites(df_13C)

    preds_1H, preds_13C = [], []
    for _ in range(n_essais):
        s1H  = generer_spectre_1H(df_1H,  pic_solvant_1H,  augmentation=True)
        s13C = generer_spectre_13C(df_13C, pic_solvant_13C, augmentation=True)

        s1H  = s1H[np.newaxis, ..., np.newaxis].astype(np.float32)
        s13C = s13C[np.newaxis, ..., np.newaxis].astype(np.float32)

        preds_1H.append(model_1H(s1H,  training=False).numpy()[0])
        preds_13C.append(model_13C(s13C, training=False).numpy()[0])

    pred_1H  = np.mean(preds_1H,  axis=0)
    pred_13C = np.mean(preds_13C, axis=0)

    pred_1H_aligned  = pred_1H[idx_1H_communes]
    pred_13C_aligned = pred_13C[idx_13C_communes]

    pred_fusion = poids_1H * pred_1H_aligned + poids_13C * pred_13C_aligned
    pred_fusion /= pred_fusion.sum()

    index_gagnant = np.argmax(pred_fusion)

    print(f"Solvant             : {solvant}  |  {n_essais} tirages moyennés")
    print(f"Prédiction 1H seul  : {le_1H.classes_[np.argmax(pred_1H)]} ({np.max(pred_1H)*100:.1f}%)")
    print(f"Prédiction 13C seul : {le_13C.classes_[np.argmax(pred_13C)]} ({np.max(pred_13C)*100:.1f}%)")
    print(f"Prédiction FUSION   : {molecules_communes[index_gagnant]} ({pred_fusion[index_gagnant]*100:.2f}%)")
    print("\nTop 5 fusion :")
    for i in np.argsort(pred_fusion)[::-1][:5]:
        print(f"  {molecules_communes[i]:<35} {pred_fusion[i]*100:.2f}%")

    return pred_fusion

# Exemple — DCM mesuré dans CDCl3
predire_fusion(
    '/content/drive/MyDrive/IA RMN/Molecules CSV 1H test/DCM.csv',
    '/content/drive/MyDrive/IA RMN/Molecules CSV 13C test/DCM.csv',
    solvant='cdcl3'
)

Solvant             : cdcl3  |  10 tirages moyennés
Prédiction 1H seul  : DCM (53.5%)
Prédiction 13C seul : DCM (47.6%)
Prédiction FUSION   : DCM (50.57%)

Top 5 fusion :
  DCM                                 50.57%
  Methanol                            17.46%
  Vinylidene_chloride                 16.72%
  Chloroacetaldehyde                  7.09%
  Urea                                6.49%


array([6.3251301e-24, 3.2131955e-23, 4.6037572e-18, ..., 1.2980550e-35,
       7.7815105e-29, 7.9818411e-23], dtype=float32)

In [ ]:
n_test = 30
all_p1H_a, all_p13C_a, all_labels = [], [], []

print(f"Traitement de {len(molecules_communes)} molécules ({n_test} tests chacune)...")

for idx_vrai, nom in enumerate(molecules_communes):
    data_1H  = molecules_1H[nom]
    data_13C = molecules_13C[nom]

    batch_s1H  = np.array([generer_spectre_1H(data_1H['df'],  data_1H['solvant'],  True)
                           for _ in range(n_test)], dtype=np.float32)[..., np.newaxis]
    batch_s13C = np.array([generer_spectre_13C(data_13C['df'], data_13C['solvant'], True)
                           for _ in range(n_test)], dtype=np.float32)[..., np.newaxis]

    p1H  = model_1H.predict(batch_s1H,  batch_size=n_test, verbose=0)
    p13C = model_13C.predict(batch_s13C, batch_size=n_test, verbose=0)

    all_p1H_a.append(p1H[:,  idx_1H_communes])
    all_p13C_a.append(p13C[:, idx_13C_communes])
    all_labels.extend([idx_vrai] * n_test)

    if (idx_vrai + 1) % 100 == 0:
        print(f"  {idx_vrai + 1}/{len(molecules_communes)} molécules traitées")

p1H_a  = np.concatenate(all_p1H_a,  axis=0)
p13C_a = np.concatenate(all_p13C_a, axis=0)
all_labels = np.array(all_labels)
del all_p1H_a, all_p13C_a
import gc; gc.collect()

acc_1H  = np.mean(np.argmax(p1H_a,  axis=1) == all_labels)
acc_13C = np.mean(np.argmax(p13C_a, axis=1) == all_labels)
print(f"\nAccuracy 1H seul  : {acc_1H:.4f}")
print(f"Accuracy 13C seul : {acc_13C:.4f}")

print("\nRecherche du meilleur poids de fusion :")
best_acc, best_w = 0, 0.5
for w in np.arange(0.0, 1.01, 0.1):
    p_fusion = w * p1H_a + (1 - w) * p13C_a
    acc = np.mean(np.argmax(p_fusion, axis=1) == all_labels)
    print(f"  poids_1H={w:.1f} / poids_13C={1-w:.1f}  →  {acc:.4f}")
    if acc > best_acc:
        best_acc, best_w = acc, w

print(f"\nMeilleure fusion : poids_1H={best_w:.1f}, accuracy = {best_acc:.4f}")
print(f"Gain vs meilleur modèle seul : {best_acc - max(acc_1H, acc_13C):+.4f}")

Traitement de 1235 molécules (30 tests chacune)...
  100/1235 molécules traitées
  200/1235 molécules traitées
  300/1235 molécules traitées
  400/1235 molécules traitées
  500/1235 molécules traitées
  600/1235 molécules traitées
  700/1235 molécules traitées
  800/1235 molécules traitées
  900/1235 molécules traitées
  1000/1235 molécules traitées
  1100/1235 molécules traitées
  1200/1235 molécules traitées

Accuracy 1H seul  : 0.9284
Accuracy 13C seul : 0.9765

Recherche du meilleur poids de fusion :
  poids_1H=0.0 / poids_13C=1.0  →  0.9765
  poids_1H=0.1 / poids_13C=0.9  →  0.9785
  poids_1H=0.2 / poids_13C=0.8  →  0.9809
  poids_1H=0.3 / poids_13C=0.7  →  0.9834
  poids_1H=0.4 / poids_13C=0.6  →  0.9868
  poids_1H=0.5 / poids_13C=0.5  →  0.9919
  poids_1H=0.6 / poids_13C=0.4  →  0.9829
  poids_1H=0.7 / poids_13C=0.3  →  0.9717
  poids_1H=0.8 / poids_13C=0.2  →  0.9592
  poids_1H=0.9 / poids_13C=0.1  →  0.9409
  poids_1H=1.0 / poids_13C=0.0  →  0.9284

Meilleure fusion : poids_1H

In [ ]:
from sklearn.metrics import confusion_matrix

p_fusion = best_w * p1H_a + (1 - best_w) * p13C_a
Y_pred_fusion = np.argmax(p_fusion, axis=1)

print(f"Accuracy 1H     : {acc_1H:.4f}")
print(f"Accuracy 13C    : {acc_13C:.4f}")
print(f"Accuracy FUSION : {np.mean(Y_pred_fusion == all_labels):.4f}")

def top_k_accuracy(y_true, proba, k):
    top_k = np.argsort(proba, axis=1)[:, -k:]
    return np.mean([y_true[i] in top_k[i] for i in range(len(y_true))])

for k in [1, 3, 5]:
    print(f"Top-{k} fusion : {top_k_accuracy(all_labels, p_fusion, k):.4f}")

if len(molecules_communes) <= 60:
    cm = confusion_matrix(all_labels, Y_pred_fusion, labels=range(len(molecules_communes)))
    fig = go.Figure(data=go.Heatmap(
        z=cm, x=molecules_communes, y=molecules_communes,
        colorscale='Blues', text=cm, texttemplate="%{text}"
    ))
    fig.update_layout(title="Matrice de confusion - Fusion 1H + 13C",
                      xaxis_title="Prédiction", yaxis_title="Vraie classe",
                      width=900, height=900)
    fig.show()
else:
    print(f"\n{len(molecules_communes)} classes — matrice trop grande, on regarde les confusions principales :")
    from collections import Counter
    erreurs = Counter()
    for i in np.where(Y_pred_fusion != all_labels)[0]:
        erreurs[(molecules_communes[all_labels[i]], molecules_communes[Y_pred_fusion[i]])] += 1
    for (vrai, pred), c in erreurs.most_common(20):
        print(f"  {c:3d}×  {vrai[:30]:<32} → {pred[:30]}")

Accuracy 1H     : 0.9284
Accuracy 13C    : 0.9765
Accuracy FUSION : 0.9919
Top-1 fusion : 0.9919
Top-3 fusion : 0.9975
Top-5 fusion : 0.9992

1235 classes — matrice trop grande, on regarde les confusions principales :
   23×  DMSO                             → Dimethyl_sulfoxide
   15×  Acetone                          → Acetonitrile
   14×  Dichloroacetonitrile             → Aminoacetonitrile
   14×  Glycolic_acid                    → 3-Oxoalanine
   13×  Diethyl_cyanophosphonate         → 2-Cyanoethyl_dihydrogen_phosph
   13×  Dioxane                          → Glycerol
   11×  1_2-Dichloroethane               → DCM
   11×  Aminoacetonitrile                → Dichloroacetonitrile
    9×  1_2-Dichloroethane               → MALONONITRILE
    9×  Dioxane                          → Dihydroxyacetone
    8×  1_2-Dibromoethane                → MALONONITRILE
    8×  _1R_-1-Amino-2-sulfanylethanes   → 2_4-Pentanedione
    7×  Vinylidene_chloride              → Trichloroethylene
    6×  1_2-Dib